# Explore here

In [1]:
# Paso 1 - Descargar el dataset y guardarlo localmente

import pandas as pd
import os

# 1. Definimos el nombre de la carpeta y la ruta del archivo
folder_name = "data"
file_path = os.path.join(folder_name, "adult_census_income.csv")

# 2. Creamos la carpeta si no existe
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"📁 Carpeta '{folder_name}' creada con éxito.")
else:
    print(f"✅ La carpeta '{folder_name}' ya existe.")

# 3. URL del dataset
url = "https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"

# 4. Descargamos y guardamos localmente
# Usamos index=False para no guardar la columna de índices de pandas
try:
    df = pd.read_csv(url)
    df.to_csv(file_path, index=False)
    print(f"💾 Dataset guardado en: {file_path}")
except Exception as e:
    print(f"❌ Error al descargar los datos: {e}")

# Una mirada rápida para confirmar que todo está bien
print(df.head())

✅ La carpeta 'data' ya existe.
💾 Dataset guardado en: data/adult_census_income.csv
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356    

In [3]:
# Paso 2 procesar los datos y preparar el dataset para el entrenamiento del modelo de recomendación

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Limpieza: Reemplazamos los '?' que se ven en tu imagen por nulos y eliminamos
df = df.replace('?', np.nan).dropna()

# 2. Encoding de la variable objetivo
le = LabelEncoder()
df['income_encoded'] = le.fit_transform(df['income'])

# 3. Variables categóricas (Ajustadas con PUNTOS según tu imagen)
categorical_cols = [
    'workclass', 'education', 'marital.status', 'occupation', 
    'relationship', 'race', 'sex', 'native.country'
]
df_final = pd.get_dummies(df, columns=categorical_cols)

# 4. Variables numéricas (También ajustadas con PUNTOS)
numerical_cols = [
    'age', 'fnlwgt', 'education.num', 'capital.gain', 
    'capital.loss', 'hours.per.week'
]

scaler = StandardScaler()
df_final[numerical_cols] = scaler.fit_transform(df_final[numerical_cols])

print("✅ ¡Logrado! Paso 2 completado sin errores.")
print(f"Ahora tienes {df_final.shape[1]} columnas listas para el sistema de recomendación.")

✅ ¡Logrado! Paso 2 completado sin errores.
Ahora tienes 106 columnas listas para el sistema de recomendación.


In [5]:
# Paso 3 y 4 Definir el modelo de recomendación y entrenarlo

from sklearn.neighbors import NearestNeighbors

# 1. Filtramos los exitosos
df_exitosos = df_final[df_final['income_encoded'] == 1].copy()

# 2. Entrenamos el buscador
model_recomender = NearestNeighbors(n_neighbors=3, metric='cosine')

# --- AQUÍ ESTÁ EL CAMBIO ---
# Quitamos tanto 'income_encoded' como la columna original 'income' que tiene el texto
X_entrenamiento = df_exitosos.drop(['income_encoded', 'income'], axis=1)
model_recomender.fit(X_entrenamiento)
# ---------------------------

print("🎯 ¡Motor de recomendación listo!")
print(f"Columnas procesadas: {X_entrenamiento.shape[1]}")


🎯 ¡Motor de recomendación listo!
Columnas procesadas: 104


In [6]:
# Paso 5 pruebas y recomendaciones

# 1. Seleccionamos un usuario de prueba (puedes cambiar el índice para probar con otros)
def recomendar_estrategia(indice_usuario):
    # 1. Obtenemos los datos del usuario que queremos ayudar
    usuario_original = df.iloc[indice_usuario]
    
    # Preparamos su perfil para el modelo (sin las columnas de ingreso)
    perfil_usuario = df_final.iloc[[indice_usuario]].drop(['income', 'income_encoded'], axis=1)
    
    # 2. Encontramos a sus 3 "vecinos" exitosos
    distancias, indices = model_recomender.kneighbors(perfil_usuario)
    
    # 3. Recuperamos la información original de esos vecinos
    vecinos_exitosos = df.iloc[df_exitosos.index[indices[0]]]
    
    # --- INTERPRETACIÓN ---
    print(f"📊 ANALIZANDO PERFIL (Índice {indice_usuario})")
    print(f"Actual: {usuario_original['age']} años, {usuario_original['education']}, {usuario_original['occupation']}")
    print(f"Ingreso actual: {usuario_original['income']}")
    print("-" * 50)
    print("💡 ESTRATEGIAS SUGERIDAS PARA SUPERAR LOS 50K:")
    
    # Vemos qué tienen los vecinos que el usuario no tiene
    sugerencias = vecinos_exitosos[['education', 'occupation', 'hours.per.week']].drop_duplicates()
    
    return sugerencias

# PROBEMOS CON ALGUIEN QUE GANE <=50K
# El índice 0 o el 4 suelen ser buenos ejemplos en este dataset
recomendar_estrategia(4)



📊 ANALIZANDO PERFIL (Índice 4)
Actual: 38 años, 10th, Adm-clerical
Ingreso actual: <=50K
--------------------------------------------------
💡 ESTRATEGIAS SUGERIDAS PARA SUPERAR LOS 50K:


,education,occupation,hours.per.week
34,HS-grad,Exec-managerial,50
51,HS-grad,Sales,72
69,Prof-school,Prof-specialty,60


In [ ]:
# Paso 5 - Prueba A 

# Buscamos a alguien con esas características
filtro_joven = df[(df['age'] == 25) & 
                  (df['education'] == 'HS-grad') & 
                  (df['hours.per.week'] <= 25) & 
                  (df['income'] == '<=50K')]

if not filtro_joven.empty:
    indice_joven = filtro_joven.index[0]
    print("✅ Usuario encontrado para la Prueba A.")
    display(recomendar_estrategia(indice_joven))
else:
    print("No encontré a alguien idéntico, probemos con el primer usuario de 25 años disponible:")
    indice_joven = df[df['age'] == 25].index[0]
    display(recomendar_estrategia(indice_joven))

✅ Usuario encontrado para la Prueba A.
📊 ANALIZANDO PERFIL (Índice 1114)
Actual: 26 años, Masters, Prof-specialty
Ingreso actual: <=50K
--------------------------------------------------
💡 ESTRATEGIAS SUGERIDAS PARA SUPERAR LOS 50K:


,education,occupation,hours.per.week
1434,7th-8th,Exec-managerial,40
196,Bachelors,Adm-clerical,40
190,Some-college,Exec-managerial,40


In [8]:
# Paso 5- Prueba B

# Buscamos a alguien con mucha experiencia pero ingresos bajos
filtro_senior = df[(df['age'] > 45) & (df['income'] == '<=50K')]

indice_senior = filtro_senior.index[0]
print(f"✅ Analizando a un usuario con experiencia (Índice {indice_senior}).")
display(recomendar_estrategia(indice_senior))

✅ Analizando a un usuario con experiencia (Índice 1).
📊 ANALIZANDO PERFIL (Índice 1)
Actual: 54 años, 7th-8th, Machine-op-inspct
Ingreso actual: <=50K
--------------------------------------------------
💡 ESTRATEGIAS SUGERIDAS PARA SUPERAR LOS 50K:


,education,occupation,hours.per.week
51,HS-grad,Sales,72
37,Prof-school,Prof-specialty,50
208,Assoc-acdm,Other-service,18


In [9]:
# Buscamos dos perfiles similares en todo excepto el sexo
prueba_mujer = df[(df['sex'] == 'Female') & (df['income'] == '<=50K')].index[10]
prueba_hombre = df[(df['sex'] == 'Male') & (df['income'] == '<=50K')].index[10]

print("👩 RECOMENDACIÓN PARA MUJER:")
display(recomendar_estrategia(prueba_mujer))

print("\n👨 RECOMENDACIÓN PARA HOMBRE:")
display(recomendar_estrategia(prueba_hombre))

👩 RECOMENDACIÓN PARA MUJER:
📊 ANALIZANDO PERFIL (Índice 149)
Actual: 46 años, Assoc-acdm, Adm-clerical
Ingreso actual: >50K
--------------------------------------------------
💡 ESTRATEGIAS SUGERIDAS PARA SUPERAR LOS 50K:


,education,occupation,hours.per.week
189,HS-grad,Prof-specialty,40
51,HS-grad,Sales,72
61,11th,Exec-managerial,40



👨 RECOMENDACIÓN PARA HOMBRE:
📊 ANALIZANDO PERFIL (Índice 127)
Actual: 59 años, HS-grad, Farming-fishing
Ingreso actual: <=50K
--------------------------------------------------
💡 ESTRATEGIAS SUGERIDAS PARA SUPERAR LOS 50K:


,education,occupation,hours.per.week
219,1st-4th,Transport-moving,18
207,Some-college,Adm-clerical,40
500,Masters,Prof-specialty,44


In [ ]:
# Analisis de resultados

# Prueba A: 26 años, Masters, Prof-specialty (< 50K$) El sistema detecta que, aunque ya tiene estudios altos, sus "vecinos exitosos" están en roles de Gestión (Exec-managerial). La recomendación aquí no es estudiar más, sino cambiar hacia el liderazgo.
# Prueba B: 54 años, 7mo-8vo grado, operario (< 50K$) Aquí el modelo es brutalmente honesto: para superar los 50K con poca educación, los vecinos exitosos o estudiaron mucho más (Prof-school) o trabajan una cantidad enorme de horas (72 horas/semana).
# Prueba C: Prueba Mujer 46 años, técnica, administrativa ($>50K$) Al ya ganar bien, el sistema muestra que otros perfiles similares lo logran mediante ventas (Sales) o especialización técnica, incluso con menos estudios formales.
# Pruebas C: Prueba Hombre 59 años, secundaria, agricultura ($\le 50K$) La recomendación es diversa: desde obtener un Masters para entrar en sectores profesionales, hasta un cambio radical hacia el Transporte, donde el ingreso suele ser más alto para ese perfil.

In [10]:
# Paso 6 - Guardar el modelo y el escalador para uso futuro
import joblib
import os 

# 1. Crear una carpeta para los modelos
if not os.path.exists('models'):
    os.makedirs('models')

# 2. Guardar el modelo y el escalador
joblib.dump(model_recomender, 'models/recomender_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print("💾 ¡Modelo y Escalador guardados con éxito!")

💾 ¡Modelo y Escalador guardados con éxito!
